## Importando os dados da camada silver
Pega os arquivos em .parquet e transforma em dataframes


In [43]:
import pandas as pd
from colorama import Fore, Style, init

alunos = pd.read_parquet(r'tech-challenge-alfabetizacao\data\silver\alunos\alunos.parquet')
meta_alfabetizacao_brasil = pd.read_parquet(r'tech-challenge-alfabetizacao\data\silver\meta_alfabetizacao_brasil\meta_alfabetizacao_brasil.parquet')
meta_alfabetizacao_municipio = pd.read_parquet(r'tech-challenge-alfabetizacao\data\silver\meta_alfabetizacao_municipio\meta_alfabetizacao_municipio.parquet')
meta_alfabetizacao_uf = pd.read_parquet(r'tech-challenge-alfabetizacao\data\silver\meta_alfabetizacao_uf')
municipio = pd.read_parquet(r'tech-challenge-alfabetizacao\data\silver\municipio\municipio.parquet')
uf = pd.read_parquet(r'tech-challenge-alfabetizacao\data\silver\uf\uf.parquet')


mapping_municipio = pd.read_csv(r'tech-challenge-alfabetizacao\src\techchallenge\config\mappings\municipios.csv')
mapping_uf = pd.read_csv(r'tech-challenge-alfabetizacao\src\techchallenge\config\mappings\uf.csv')
mapping_rede = pd.read_csv(r'tech-challenge-alfabetizacao\src\techchallenge\config\mappings\rede.csv')


GOLD_PATH = r'tech-challenge-alfabetizacao\data\gold'

Essa agregação isola os 5 municípios com os piores índices de alfabetização de cada estado, servindo como uma tabela de "Alerta Vermelho" para direcionamento de recursos.

In [ ]:
def gerar_top_5_municipios_criticos_por_uf(ano:int):

    # Seleciona as colunas que vão compor a agregação e juntando as tabelas fato e dimensão

    colunas_selecionadas = ['taxa_alfabetizacao','Municipio', 'UF']
    df_municipio_ano_selecionado = municipio[municipio['ano'].astype(int) == int(ano)].copy()
    

    # Verificar se o ano existe informações do ano na base de dados 
    
    if df_municipio_ano_selecionado.empty:
        print(f"{Fore.YELLOW}Aviso: Nenhum dado foi encontrado para o ano {ano}.")
        return None

    # Agregação: Ordenar por UF e Taxa (Crescente) e pegar os 5 primeiros de cada estado

    top_5_municipios_criticos_ano = (df_municipio_ano_selecionado.sort_values(
        by=['UF', 'taxa_alfabetizacao'], ascending = [True, True])
        .groupby('UF')
        .head(5)
    )[colunas_selecionadas]
    top_5_municipios_criticos_ano.reset_index(drop=True, inplace=True)
    
    # Salvar o resultado na Camada Gold incluindo o ano no nome do arquivo (boa prática de versionamento)
    output_file = f"{GOLD_PATH}/top5_municipios_criticos_por_estado_{ano}.parquet"
    top_5_municipios_criticos_ano.to_parquet(output_file, index=False)
    
    print(f"{Fore.GREEN}✔ Agregação do ano {ano} concluída com sucesso!")
    print(f"{Fore.YELLOW}Salvo em: {output_file}")
    print(f"Total de registros gerados: {len(top_5_municipios_criticos_ano)}")
    
    return top_5_municipios_criticos_ano

if __name__ == "__main__":
    # Executa a função passando o ano desejado como parâmetro para teste
    ano_analise = 2023
    df_criticos = gerar_top_5_municipios_criticos_por_uf(ano=ano_analise)
    
    if df_criticos is not None:
        print(f"\nExemplo dos dados gerados para {ano_analise} (Primeiros 10 registros):")
        print(df_criticos.head(10))
    

✔ Agregação do ano 2023 concluída com sucesso!
Salvo em: tech-challenge-alfabetizacao\data\gold/top5_municipios_criticos_por_estado_2023.parquet
Total de registros gerados: 125

Exemplo dos dados gerados para 2023 (Primeiros 10 registros):
   taxa_alfabetizacao            Municipio  UF
0               48.61             Barcelos  AC
1               54.48             Barcelos  AC
2               77.48             Barcelos  AC
3               13.27        Joaquim Gomes  AL
4               14.29               Inhapi  AL
5               20.41          Ouro Branco  AL
6               20.41          Ouro Branco  AL
7               20.82  Palmeira dos Índios  AL
8               13.39               Uarini  AM
9               19.19             Eirunepé  AM


Essa agregação compara a taxa de alfabetização com a meta estipulada para o ano

In [54]:
def gerar_analise_meta_por_uf(ano):
    ano_analisado = 2024
    col_meta = f'meta_alfabetizacao_{ano_analisado}'
    compara_meta_taxa_uf = meta_alfabetizacao_uf[meta_alfabetizacao_uf['ano'] == ano_analisado][['sigla_uf', col_meta, 'taxa_alfabetizacao']].copy()

    def avaliar_meta(row):
        taxa = row['taxa_alfabetizacao']
        meta = row[col_meta]
        
        if pd.isna(taxa) or pd.isna(meta):
            return "Sem Dados"
        elif taxa >= meta:
            return "Acima da Meta"
        else:
            return "Abaixo da Meta"

    # Aplicamos a função linha por linha usando axis=1
    compara_meta_taxa_uf['status_meta'] = compara_meta_taxa_uf.apply(avaliar_meta, axis=1)
    
    # Salvar o resultado na Camada Gold incluindo o ano no nome do arquivo (boa prática de versionamento)
    output_file = f"{GOLD_PATH}/analise_meta_versus_taxa_por_uf_{ano}.parquet"
    compara_meta_taxa_uf.to_parquet(output_file, index=False)
    
    print(f"{Fore.GREEN}✔ Agregação do ano {ano} concluída com sucesso!")
    print(f"{Fore.YELLOW}Salvo em: {output_file}")
    print(f"Total de registros gerados: {len(compara_meta_taxa_uf)}")
        
    return compara_meta_taxa_uf

if __name__ == "__main__":
    # Executa a função passando o ano desejado como parâmetro para teste
    ano_analise = 2024
    df_meta_versus_taxa = gerar_analise_meta_por_uf(ano=ano_analise)
    
    if df_meta_versus_taxa is not None:
        print(f"\nExemplo dos dados gerados para {ano_analise} (Primeiros 10 registros):")
        print(df_meta_versus_taxa.head(10))


✔ Agregação do ano 2024 concluída com sucesso!
Salvo em: tech-challenge-alfabetizacao\data\gold/analise_meta_versus_taxa_por_uf_2024.parquet
Total de registros gerados: 27

Exemplo dos dados gerados para 2024 (Primeiros 10 registros):
   sigla_uf  meta_alfabetizacao_2024  taxa_alfabetizacao     status_meta
0        RR                      NaN                 NaN       Sem Dados
2        SE                     38.3               38.39   Acima da Meta
8        BA                     43.4               35.96  Abaixo da Meta
9        RN                     43.8               39.29  Abaixo da Meta
11       AP                     47.6               46.62  Abaixo da Meta
16       AL                     49.7               48.63  Abaixo da Meta
17       TO                     49.5               50.07   Acima da Meta
22       AC                      NaN               51.38       Sem Dados
25       MS                     52.8               55.87   Acima da Meta
28       PA                     53.

Indicador de alfabetização por município, no ano desejado

In [ ]:
def indicador_alfabetizacao_municipio (ano:int) :

    # Seleciona as colunas que vão compor a agregação

    colunas_selecionadas = ['Municipio', 'UF', 'Rede','taxa_alfabetizacao']
    df_municipio_ano_selecionado = municipio[municipio['ano'].astype(int) == int(ano)].copy()
    indicador_alfabetizacao_por_municipio = df_municipio_ano_selecionado[colunas_selecionadas]

    # Verificar se o ano existe informações do ano na base de dados 
    
    if df_municipio_ano_selecionado.empty:
        print(f"{Fore.YELLOW}Aviso: Nenhum dado foi encontrado para o ano {ano}.")
        return None
    
    # Salvar o resultado na Camada Gold incluindo o ano no nome do arquiv
    output_file = f"{GOLD_PATH}/lista_indicador_alfabetizacao_por_municipio_{ano}.parquet"
    indicador_alfabetizacao_por_municipio.to_parquet(output_file, index=False)
    
    print(f"{Fore.GREEN}✔ Agregação do ano {ano} concluída com sucesso!")
    print(f"{Fore.YELLOW}Salvo em: {output_file}")
    print(f"Total de registros gerados: {len(indicador_alfabetizacao_por_municipio)}")
    
    return indicador_alfabetizacao_por_municipio

if __name__ == "__main__":
    # Executa a função passando o ano desejado como parâmetro para teste
    ano_analise = 2024
    df_indicador_por_municipio = indicador_alfabetizacao_municipio(ano=ano_analise)
    
    if df_criticos is not None:
        print(f"\nExemplo dos dados gerados para {ano_analise} (Primeiros 10 registros):")
        print(df_indicador_por_municipio.head(10))

✔ Agregação do ano 2024 concluída com sucesso!
Salvo em: tech-challenge-alfabetizacao\data\gold/lista_indicador_alfabetizacao_por_municipio_2024.parquet
Total de registros gerados: 12448

Exemplo dos dados gerados para 2024 (Primeiros 10 registros):
                   Municipio  UF                            Rede  \
11547  Senador Elói de Souza  RN                        Estadual   
11548        Augusto Pestana  RS                        Estadual   
11549  Gramado dos Loureiros  RS                       Municipal   
11550                 Jundiá  RN  Pública (Estadual e Municipal)   
11551              Buerarema  BA                        Estadual   
11552               Quevedos  RS  Pública (Estadual e Municipal)   
11553             Pau Brasil  BA                        Estadual   
11554               Quevedos  RS                        Estadual   
11555       Chapada de Areia  TO  Pública (Estadual e Municipal)   
11556            Verdelândia  MG                       Municipal   

 